# Chat Models and Messages

LangChain chains ultimately call a **chat model** — a Runnable that accepts **messages** and returns an **`AIMessage`**. Prompt templates produce those messages; output parsers consume the model's reply. Understanding the message contract and model configuration is essential before building prompts (**04**), parsers (**05**), or RAG pipelines (**11**).

This notebook covers the chat-model paradigm (vs legacy text-completion), every core **message type**, invoking **`ChatOpenAI`**, reading **`AIMessage`** fields (content, `tool_calls`, usage metadata), model parameters (`temperature`, `max_tokens`, …), **`init_chat_model`** for provider switching, multi-turn message lists, streaming at the model layer, structured output binding, and test doubles for development without API calls.

**02. Runnable Protocol and LCEL** established how models fit in piped chains. Here we focus on what happens **inside** the LLM step.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import asyncio
import json

import langchain_core
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

print("langchain-core:", langchain_core.__version__)

---

## 1. Chat Models vs Completion Models

Early LLM APIs accepted a single string prompt and returned a string completion. Modern APIs use a **chat completions** format: an ordered list of **role-tagged messages**.

| Era | API shape | LangChain class |
|-----|-----------|-----------------|
| Legacy completion | `prompt: str → text: str` | `LLM` (deprecated) |
| Modern chat | `messages: list → AIMessage` | `BaseChatModel` / `ChatOpenAI` |

```
Completion (old):  "Translate to French: hello"  →  "bonjour"

Chat (modern):     [SystemMessage(...), HumanMessage(...)]  →  AIMessage("bonjour")
```

Every major provider (OpenAI, Anthropic, Google) exposes chat endpoints. LangChain 1.x targets **`BaseChatModel`** implementations exclusively in new application code.

---

## 2. The Message Abstraction

A **message** is a structured turn in a conversation. Each message has:

| Field | Description |
|-------|-------------|
| **`type`** | Role string: `system`, `human`, `ai`, `tool`, … |
| **`content`** | Text, or multimodal blocks (text + image URLs in advanced use) |
| **`id`** | Optional unique identifier |
| **`additional_kwargs`** | Provider-specific extras |
| **`response_metadata`** | Token usage, model name, finish reason (on AI messages) |

Chat models read the **full message list** as context — the model sees prior turns when generating the next reply.

---

## 3. Core Message Types

| Class | `type` | Role | When to use |
|-------|--------|------|-------------|
| **`SystemMessage`** | `system` | Instructions | Persona, rules, output format |
| **`HumanMessage`** | `human` | User | End-user input |
| **`AIMessage`** | `ai` | Assistant | Model output; may include `tool_calls` |
| **`ToolMessage`** | `tool` | Tool result | Return value after executing a tool |

```
Turn 1:  [System]  You are a concise API expert.
         [Human]   What is JWT?
         [AI]      JSON Web Token for auth...

Turn 2:  [Human]   What header carries it?
         [AI]      Authorization: Bearer ...

Tool:    [AI]      (tool_calls: get_weather)
         [Tool]    {"temp": 72}
         [AI]      It is 72°F.
```

Tool messages are covered in depth in **12. Tools and Tool Calling**; we preview them here for completeness.

In [ ]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
)

conversation = [
    SystemMessage(content="You are a patient teacher. Use short sentences."),
    HumanMessage(content="What is a REST API?"),
    AIMessage(content="A REST API exposes resources over HTTP using standard methods like GET and POST."),
    HumanMessage(content="Give one example resource."),
]

for i, msg in enumerate(conversation, 1):
    preview = msg.content[:70] + "..." if len(msg.content) > 70 else msg.content
    print(f"{i}. {msg.__class__.__name__:15s} type={msg.type!r:8s}  {preview}")

### 3.1 Tuple Shorthand in Prompts

Prompt templates accept `(role, content)` tuples — converted to message objects internally. Roles map to the classes above: `"system"`, `"human"`, `"ai"`. Full template syntax is in **04. Prompt Templates**.

---

## 4. ChatOpenAI — The Primary Model Class

**`ChatOpenAI`** from `langchain-openai` wraps OpenAI's chat completions API. It implements **`BaseChatModel`** and therefore the full **Runnable** API (`invoke`, `stream`, `batch`, async variants).

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0.2,
    max_tokens=256,
)

print("model:", llm.model_name)
print("temperature:", llm.temperature)
print("is Runnable:", hasattr(llm, "invoke"), hasattr(llm, "stream"))

### 4.1 Common Constructor Parameters

| Parameter | Effect |
|-----------|--------|
| **`model`** | Model ID (`gpt-4o-mini`, `gpt-4o`, …) |
| **`temperature`** | Randomness (0 = deterministic-ish, higher = creative) |
| **`max_tokens`** | Cap on completion length |
| **`top_p`** | Nucleus sampling alternative to temperature |
| **`timeout`** | HTTP timeout in seconds |
| **`max_retries`** | Automatic retry on transient API errors |
| **`streaming`** | Default streaming behavior |
| **`model_kwargs`** | Extra kwargs forwarded to OpenAI API |

---

## 5. Invoking a Chat Model

### 5.1 With a Message List (Canonical)

In [ ]:
messages = [
    SystemMessage(content="Reply in at most 12 words."),
    HumanMessage(content="What does HTTP stand for?"),
]

response: AIMessage = llm.invoke(messages)

print("type:", type(response).__name__)
print("content:", response.content)
print("type field:", response.type)

### 5.2 With a Plain String (Shortcut)

Passing a `str` is converted to a single `HumanMessage`. Useful for quick tests; production chains should use explicit messages or prompt templates.

In [ ]:
shortcut_response = llm.invoke("Say 'pong' and nothing else.")
print(shortcut_response.content)

### 5.3 Multi-Turn — Manual History

Append each `HumanMessage` and `AIMessage` to the list. The model sees full context on every call. Persistent session management is covered in **14. Memory and Chat History**.

In [ ]:
history = [SystemMessage(content="You help with API documentation. Be brief.")]

def chat(user_text: str) -> str:
    history.append(HumanMessage(content=user_text))
    ai = llm.invoke(history)
    history.append(ai)
    return ai.content


print("Turn 1:", chat("What is OAuth2 in one sentence?"))
print("Turn 2:", chat("What problem does it solve?"))
print("message count:", len(history))

---

## 6. Anatomy of an AIMessage

The model returns an **`AIMessage`**. Important fields:

In [ ]:
ai = llm.invoke([
    SystemMessage(content="You are helpful."),
    HumanMessage(content="Name two HTTP methods."),
])

print("content:", ai.content)
print("tool_calls:", ai.tool_calls)
print("invalid_tool_calls:", ai.invalid_tool_calls)
print()
print("response_metadata:", json.dumps(ai.response_metadata, indent=2, default=str))

### 6.1 response_metadata — Tokens and Finish Reason

| Key | Meaning |
|-----|--------|
| `token_usage` | `prompt_tokens`, `completion_tokens`, `total_tokens` |
| `model_name` | Model that served the request |
| `finish_reason` | `stop`, `length`, `tool_calls`, … |

Use token counts for cost estimation and debugging truncated outputs (`finish_reason: length`).

In [ ]:
usage = ai.response_metadata.get("token_usage", {})
print(f"prompt={usage.get('prompt_tokens', 0)} completion={usage.get('completion_tokens', 0)} total={usage.get('total_tokens', 0)}")
print("finish_reason:", ai.response_metadata.get("finish_reason"))

---

## 7. Model Parameters in Practice

### 7.1 Temperature

In [ ]:
prompt = "Invent a one-word name for a notes API product."

cold = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
warm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=1.0)

print("temp=0:", cold.invoke(prompt).content)
print("temp=1:", warm.invoke(prompt).content)

### 7.2 max_tokens — Prevent Runaway Completions

In [ ]:
short_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, max_tokens=8, temperature=0)
long_request = HumanMessage(content="Explain microservices in detail with examples.")

truncated = short_llm.invoke([long_request])
print("content:", truncated.content)
print("finish_reason:", truncated.response_metadata.get("finish_reason"))

### 7.3 Runtime Overrides with bind()

`.bind()` attaches kwargs to a **single invocation** without creating a new model instance:

In [ ]:
creative = llm.bind(temperature=0.9, max_tokens=20)
strict = llm.bind(temperature=0, max_tokens=20)

q = "Describe JSON in three words."
print("creative:", creative.invoke(q).content)
print("strict:", strict.invoke(q).content)

---

## 8. init_chat_model — Provider-Agnostic Factory

LangChain 1.x provides **`init_chat_model`** to construct chat models from a string identifier:

In [ ]:
from langchain.chat_models import init_chat_model

llm_via_init = init_chat_model(
    "openai:gpt-4o-mini",
    api_key=OPENAI_API_KEY,
    temperature=0,
)

print(llm_via_init.invoke("Say 'init_chat_model works' only.").content)

Identifier format: `"provider:model"` — e.g. `"openai:gpt-4o-mini"`, `"anthropic:claude-3-5-sonnet-20241022"` (requires the matching integration package installed).

| Approach | When to use |
|----------|-------------|
| **`ChatOpenAI(...)`** | OpenAI-only apps; full parameter control |
| **`init_chat_model(...)`** | Multi-provider config, env-driven model selection |

---

## 9. Streaming at the Model Layer

`llm.stream(messages)` yields **`AIMessageChunk`** objects. Concatenate `.content` for the full text. Chains use `chain.stream()` (**07**) which propagates streaming through parsers.

In [ ]:
print("Model stream:")
parts: list[str] = []
for chunk in llm.stream([HumanMessage(content="Count from 1 to 5 slowly, one number per line.")]):
    if chunk.content:
        parts.append(chunk.content)
        print(chunk.content, end="", flush=True)
print("\n---")
print("chunk count:", len(parts))

---

## 10. Batch and Async Invocations

Chat models inherit Runnable **`batch`** and async methods. Each input is an independent message list (or string).

In [ ]:
inputs = [
    [HumanMessage(content="One word for GET.")],
    [HumanMessage(content="One word for POST.")],
    [HumanMessage(content="One word for DELETE.")],
]

batch_results = llm.batch(inputs)
for inp, out in zip(inputs, batch_results):
    print(f"{inp[0].content} → {out.content}")

In [ ]:
async def demo_model_async():
    result = await llm.ainvoke([HumanMessage(content="Say async OK.")])
    print("ainvoke:", result.content)

    print("astream:", end=" ")
    async for chunk in llm.astream([HumanMessage(content="Say stream OK.")]):
        print(chunk.content, end="", flush=True)
    print()


asyncio.run(demo_model_async())

---

## 11. Structured Output on the Model

`.with_structured_output(schema)` wraps the model to return a **Pydantic model** or dict instead of raw `AIMessage`. Parsing strategies are detailed in **05. Output Parsers and Structured Output**.

In [ ]:
from pydantic import BaseModel, Field


class HttpMethodFact(BaseModel):
    method: str = Field(description="HTTP method name")
    purpose: str = Field(description="One sentence purpose")
    idempotent: bool = Field(description="Whether the method is idempotent")


structured_llm = llm.with_structured_output(HttpMethodFact)

fact = structured_llm.invoke("Describe the PUT HTTP method.")
print(type(fact))
print(fact)

---

## 12. Tool Calls on AIMessage (Preview)

When tools are bound with `.bind_tools(...)`, the model may respond with **`tool_calls`** instead of user-facing text. Full tool execution loops are in **12. Tools and Tool Calling**.

In [ ]:
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


llm_tools = llm.bind_tools([multiply])
tool_response = llm_tools.invoke("What is 19 times 21? Use the multiply tool.")

print("content:", tool_response.content or "(empty — model chose tool)")
print("tool_calls:", tool_response.tool_calls)

### 12.1 ToolMessage — Returning Results to the Model

After executing a tool, append a **`ToolMessage`** with the result and the matching `tool_call_id`:

In [ ]:
if tool_response.tool_calls:
    tc = tool_response.tool_calls[0]
    result = multiply.invoke(tc["args"])
    follow_up_messages = [
        HumanMessage(content="What is 19 times 21? Use the multiply tool."),
        tool_response,
        ToolMessage(content=str(result), tool_call_id=tc["id"]),
    ]
    final = llm_tools.invoke(follow_up_messages)
    print("final answer:", final.content)

---

## 13. Chat Models in LCEL Chains

In a chain, the chat model sits between prompt and parser:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a concise assistant."),
        ("human", "{question}"),
    ])
    | llm
    | StrOutputParser()
)

print(chain.invoke({"question": "What is HTTPS?"}))
print("output type:", type(chain.invoke({"question": "What is TLS?"})).__name__)

Without **`StrOutputParser`**, the chain returns an **`AIMessage`**. Parsers normalize output at the chain boundary — see **05**.

---

## 14. Testing Without API Calls — FakeListChatModel

Use **`FakeListChatModel`** from `langchain-core` in unit tests and notebooks when you want deterministic responses:

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel

fake_llm = FakeListChatModel(responses=[
    "JWT uses Bearer tokens in the Authorization header.",
    "Use pytest fixtures in conftest.py.",
])

test_chain = (
    ChatPromptTemplate.from_template("Answer: {q}")
    | fake_llm
    | StrOutputParser()
)

print(test_chain.invoke({"q": "JWT?"}))
print(test_chain.invoke({"q": "pytest?"}))

---

## 15. BaseChatModel Interface

All chat model integrations implement **`BaseChatModel`**:

| Method | Returns |
|--------|--------|
| `_generate(messages)` | `ChatResult` with `generations[0].message` as `AIMessage` |
| `_stream(messages)` | Iterator of `ChatGenerationChunk` |
| `invoke` / `stream` / `batch` | Public Runnable API (calls `_generate` / `_stream`) |

Custom model wrappers (internal gateways, fine-tuned endpoints) subclass `BaseChatModel` and implement `_generate`. Most applications use provider classes (`ChatOpenAI`, etc.) directly.

---

## 16. Token Usage Visualization

Track prompt vs completion tokens across calls for cost awareness:

In [ ]:
prompts = [
    "Define API in 5 words.",
    "Define REST in 5 words.",
    "Define GraphQL in 5 words.",
]

prompt_tokens, completion_tokens = [], []

for p in prompts:
    ai = llm.invoke([HumanMessage(content=p)])
    u = ai.response_metadata.get("token_usage", {})
    prompt_tokens.append(u.get("prompt_tokens", 0))
    completion_tokens.append(u.get("completion_tokens", 0))

x = np.arange(len(prompts))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width / 2, prompt_tokens, width, label="prompt tokens", color="#4c72b0")
ax.bar(x + width / 2, completion_tokens, width, label="completion tokens", color="#55a868")
ax.set_xticks(x)
ax.set_xticklabels(["API", "REST", "GraphQL"])
ax.set_ylabel("tokens")
ax.set_title("Token usage per call (gpt-4o-mini)")
ax.legend()
plt.tight_layout()
plt.show()

---

## 17. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Using completion `LLM` classes | Deprecated API | Use `ChatOpenAI` |
| Forgetting system message | Inconsistent persona | Add `SystemMessage` or template system role |
| Not appending AI replies to history | Model loses context | Append both human and AI messages |
| Ignoring `finish_reason: length` | Truncated answers | Increase `max_tokens` or shorten prompt |
| Expecting text when tools bound | Empty `content`, `tool_calls` populated | Execute tools, append `ToolMessage` |
| Hardcoding API key in git | Security leak | Use env vars or secrets manager |
| No parser in chain | Downstream gets `AIMessage` | Add `StrOutputParser()` or structured parser |

---

## 18. Summary

LangChain chat models consume **message lists** and return **`AIMessage`**. Core types: **`SystemMessage`**, **`HumanMessage`**, **`AIMessage`**, **`ToolMessage`**. **`ChatOpenAI`** is the primary OpenAI integration; **`init_chat_model`** supports provider-agnostic configuration.

Key takeaways:

- Chat models implement **Runnable** — `invoke`, `stream`, `batch`, and async work at the model layer and inside chains.
- Inspect **`response_metadata`** for token usage and **`finish_reason`**.
- Tune **`temperature`**, **`max_tokens`**, and runtime **`.bind()`** overrides per use case.
- **`with_structured_output`** and **`bind_tools`** extend the model for parsers (**05**) and tools (**12**).
- Use **`FakeListChatModel`** for deterministic tests.

Demonstrations covered message construction, multi-turn history, streaming chunks, batch/async calls, structured output, tool call preview, LCEL integration, and token visualization.

Next: **04. Prompt Templates** — `ChatPromptTemplate`, variables, few-shot examples, and `MessagesPlaceholder` for dynamic prompts.